# Deaths per reported case across COVID-19 waves
This is an exploratory ratio analysis, **not** a case-fatality-risk estimate. Deaths often occur weeks after diagnosis, reporting is incomplete, and the case denominator changed over time. We compare raw weekly ratios, smoothed ratios, and simple lagged correlation.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/"data/processed/us_covid_clean.csv").exists(): ROOT=ROOT.parent
df=pd.read_csv(ROOT/"data/processed/us_covid_clean.csv",parse_dates=["week_ending"]).sort_values("week_ending")
waves=pd.read_csv(ROOT/"data/processed/us_covid_waves.csv",parse_dates=["start_date","end_date"])
df["death_case_ratio"]=df.new_deaths/df.new_cases.replace(0,np.nan)
df["ratio_4wk"]=df.death_case_ratio.rolling(4,min_periods=2).mean()
df.head()

## 1. Weekly deaths and cases, with smoothed ratio

In [ ]:
fig,ax=plt.subplots(figsize=(13,5))
ax.plot(df.week_ending,df.death_case_ratio,alpha=.25,label="Weekly deaths / weekly cases")
ax.plot(df.week_ending,df.ratio_4wk,linewidth=2,label="4-week rolling ratio")
ax.set(title="Weekly deaths-to-cases ratio (exploratory)",xlabel="Week ending",ylabel="Ratio")
ax.grid(alpha=.25); ax.legend(); plt.tight_layout()

## 2. Wave-level descriptive summary
Wave boundaries are taken from the supplied wave summary. For each wave, we report total reported deaths divided by total reported cases, which is less volatile than averaging weekly ratios. It still is not a cohort-based fatality risk because deaths and cases are not linked by infection date.

In [ ]:
rows=[]
for _,w in waves.iterrows():
    sub=df[(df.week_ending>=w.start_date)&(df.week_ending<=w.end_date)]
    cases=sub.new_cases.sum(); deaths=sub.new_deaths.sum()
    rows.append({"wave":int(w.wave),"start":w.start_date.date(),"end":w.end_date.date(),"cases_sum":cases,"deaths_sum":deaths,"death_to_case_ratio":deaths/cases if cases else np.nan})
wave_ratio=pd.DataFrame(rows)
wave_ratio

## 3. Lag sensitivity
We calculate Pearson correlation between weekly cases and deaths shifted by 0–8 weeks. This helps demonstrate temporal displacement, but correlation is not a causal estimate and the aggregate waves are influenced by reporting artifacts.

In [ ]:
lag_results=[]
for lag in range(9):
    aligned=pd.concat([df.new_cases,df.new_deaths.shift(-lag)],axis=1).dropna()
    corr=aligned.iloc[:,0].corr(aligned.iloc[:,1]) if len(aligned)>2 else np.nan
    lag_results.append({"death_lag_weeks":lag,"correlation":corr})
lag_df=pd.DataFrame(lag_results)
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(lag_df.death_lag_weeks,lag_df.correlation,"o-")
ax.set(title="Weekly cases vs. deaths at different forward lags",xlabel="Deaths shifted earlier by lag (weeks)",ylabel="Pearson correlation")
ax.grid(alpha=.3); plt.tight_layout()
lag_df

In [ ]:
outdir=ROOT/"results"; outdir.mkdir(exist_ok=True)
wave_ratio.to_csv(outdir/"deaths_per_case_by_wave.csv",index=False)
lag_df.to_csv(outdir/"cases_deaths_lag_correlation.csv",index=False)
print("Exported wave ratios and lag-correlation tables to",outdir)